# AI Models API — usage examples

Showcase for the unified **`src.ai_models`** layer (`ModelManager`): loading pre-trained
models, training new ones, managing several at once, and handing a model to the XAI layer.

This notebook is a *demonstration* — nothing in `src/` imports it. It uses randomly
generated data, so numbers are meaningless; only the **call patterns** matter.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / "src").exists() and (candidate / "assets").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
from src.ai_models import ModelManager

print("repo root:", REPO_ROOT)


## 1 · FastAPI integration (reference only)

Reference snippet — not executed here, since it would start a web server.

```python
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import numpy as np
from src.ai_models import ModelManager

app = FastAPI()
manager = ModelManager()

class PredictionRequest(BaseModel):
    dataset: str
    model_type: str = "mlp"
    data: list

@app.get("/models/available")
def list_models():
    return manager.list_available_pretrained()

@app.post("/predict")
def predict(request: PredictionRequest):
    try:
        model = manager.load_model(dataset=request.dataset, model_type=request.model_type)
        X = np.array([request.data])
        predictions = model.predict(X)[0]
        return {
            "dataset": request.dataset,
            "predictions": predictions.tolist(),
            "confidence": float(np.max(predictions)),
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))
```


## 2 · Discover and load a pre-trained model

Weights live in `assets/model_weights/<model_type>/<agent>/` and are discovered by the registry.


In [ ]:
manager = ModelManager()

available = manager.list_available_pretrained()
print("discovered models:", len(available["available_models"]))
print(available["available_models"][:8], "...")


> **Note.** Some shipped checkpoints were trained with a different `input_dim` than
> `_DATASET_DEFAULTS` builds today, so `load_model(...)` can raise a `state_dict` shape
> error. That is a known data/architecture mismatch, not an API problem — hence the
> `try/except` below.


In [ ]:
try:
    model = manager.load_model(dataset="wine_quality", model_type="mlp", source="coxam")
    X_test = np.random.randn(5, 11)
    predictions = manager.predict(X_test)
    print("predictions shape:", predictions.shape)
    print("model info:", model.get_info())
    print("loaded models:", manager.list_loaded_models())
except Exception as exc:
    print(f"load skipped -> {type(exc).__name__}: {exc}")


## 3 · Train a new model

`create_model(...)` then `train(...)`, `evaluate(...)`, `save_model(...)`.


In [ ]:
manager = ModelManager()

model = manager.create_model(
    dataset="custom_data",
    model_type="mlp",
    input_dim=20,
    num_classes=3,
    hidden_dimension=64,
    dropout_rate=0.2,
)

X_train = np.random.randn(1000, 20)
y_train = np.random.randint(0, 3, 1000)
X_dev = np.random.randn(200, 20)
y_dev = np.random.randint(0, 3, 200)

training_info = manager.train(X_train, y_train, X_dev=X_dev, y_dev=y_dev, epochs=5, batch_size=32)
print("training info:", training_info)

X_test = np.random.randn(100, 20)
y_test = np.random.randint(0, 3, 100)
print(f"test accuracy: {manager.evaluate(X_test, y_test):.4f}")


## 4 · Manage several models at once

`auto_activate=False` keeps a loaded model out of the active slot so you can compare backends.


In [ ]:
try:
    manager = ModelManager()
    mlp_model = manager.load_model("wine_quality", "mlp", auto_activate=False)
    xgb_model = manager.load_model("wine_quality", "xgboost", auto_activate=False)

    X_test = np.random.randn(3, 11)
    print("MLP:", mlp_model.predict(X_test))
    print("XGBoost:", xgb_model.predict(X_test))

    manager.set_active_model("wine_quality_mlp_coxam")
    print("active:", manager.get_active_model().get_info())
except Exception as exc:
    print(f"comparison skipped -> {type(exc).__name__}: {exc}")


## 5 · Hand the trained model to the XAI layer

The model produced here is exactly what `create_xai_method(..., ai_model=...)` expects.


In [ ]:
manager = ModelManager()
model = manager.create_model(dataset="wine_quality", model_type="mlp", input_dim=11, num_classes=2)

X_train = np.random.randn(1000, 11)
y_train = np.random.randint(0, 2, 1000)
X_test = np.random.randn(100, 11)
y_test = np.random.randint(0, 2, 100)

manager.train(X_train, y_train, epochs=5)
print(f"test accuracy: {manager.evaluate(X_test, y_test):.4f}")

# Ready for explanation generation via the XAI adapter:
#   from src.xai_adapter import create_xai_method
#   method = create_xai_method("shap", ai_model=model.engine, train_data=X_train)
#   result = method.fit(X_train).explain(X_test[:5])
print("ready for XAI explanation generation")
